In [1]:
# ============================================================
# S1: WEARABLE-ONLY BASELINE (no PSG, no KD, no artifact gate)
#
#   Zmax EEG ──> Zmax Encoder ──┐
#                                ├── Concat Fusion ──> Classifier
#   E4       ──> E4 Encoder ────┘
#
#   Loss = CE only (weighted by class frequency)
#
# UPDATED: now uses the 3-way split (_train_subs.npy /
# _val_subs.npy / _test_subs.npy) from make_train_val_split.py.
# Checkpoint selection uses VAL F1 (not test F1) every epoch;
# TEST is evaluated exactly once, after training, using the
# val-selected best checkpoint.
#
# This is the reference point. Once this number is in, the next
# script (S2) adds knowledge distillation from the frozen PSG
# teacher and we compare against THIS baseline to measure the
# benefit of KD.
# ============================================================

import os
import csv
import math
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
warnings.filterwarnings('ignore')

# ============================================================
# PATHS
# ============================================================
STUDENT_DATA_PATH = r"D:\22\AA\preprocess\preprocessed_student_zmax_e4"
SPLIT_PATH        = r"D:\22\AA\AA journal\preprocess\preprocessed_split_v2"   # has _train/_val/_test_subs.npy
EVAL_PATH         = r"D:\22\AA\AA journal\evaluation\teacher-student\s1_wearable_baseline_valfixed"
os.makedirs(EVAL_PATH, exist_ok=True)

LABEL_NAMES = ["Wake", "N1", "N2", "N3", "REM"]
N_EPOCHS    = 30
BATCH_SIZE  = 64
SEEDS       = [42, 123, 256, 789, 999]

D_MODEL = 96
DROPOUT = 0.3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"S1: Wearable-only baseline (Zmax EEG + E4, plain CE, no KD, no gate)")
print(f"Output : {EVAL_PATH}")


# ============================================================
# 3-WAY SPLIT (train / val / test)
# ============================================================
_train_path = os.path.join(SPLIT_PATH, "_train_subs.npy")
_val_path   = os.path.join(SPLIT_PATH, "_val_subs.npy")
_test_path  = os.path.join(SPLIT_PATH, "_test_subs.npy")

for p, name in [(_train_path, "_train_subs.npy"),
                (_val_path,   "_val_subs.npy"),
                (_test_path,  "_test_subs.npy")]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"{name} not found at {SPLIT_PATH}")

TRAIN_SUBS = np.load(_train_path, allow_pickle=True).tolist()
VAL_SUBS   = np.load(_val_path,   allow_pickle=True).tolist()
TEST_SUBS  = np.load(_test_path,  allow_pickle=True).tolist()

assert set(TRAIN_SUBS).isdisjoint(VAL_SUBS),  "TRAIN/VAL subject overlap!"
assert set(TRAIN_SUBS).isdisjoint(TEST_SUBS), "TRAIN/TEST subject overlap!"
assert set(VAL_SUBS).isdisjoint(TEST_SUBS),   "VAL/TEST subject overlap!"

print(f"Split loaded -> Train:{len(TRAIN_SUBS)}  Val:{len(VAL_SUBS)}  Test:{len(TEST_SUBS)}")


# ============================================================
# DATASET (Zmax + E4 only, single center epoch, plain labels)
# ============================================================
class WearableDataset(Dataset):
    def __init__(self, subject_list, data_path):
        self.data  = []
        self.index = []
        label_counter = Counter()

        n_loaded = 0
        for sub in subject_list:
            fp = os.path.join(data_path, f"{sub}.npz")
            if not os.path.exists(fp):
                continue
            with np.load(fp) as d:
                zmax_arr = d['zmax_eeg']    # (N, 2, 1920)
                e4_arr   = d['e4']          # (N, 3, 1920)
                labels   = d['labels'].copy()

            n = min(zmax_arr.shape[0], e4_arr.shape[0], len(labels))
            sub_idx = len(self.data)
            self.data.append((zmax_arr[:n], e4_arr[:n], labels[:n]))
            for i in range(n):
                self.index.append((sub_idx, i))
                label_counter[int(labels[i])] += 1
            n_loaded += 1

        self.label_counts = np.array(
            [label_counter[i] for i in range(5)], dtype=np.float32
        )
        print(f"  Subjects: {n_loaded}   Samples: {len(self.index):,}")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        sub_idx, i = self.index[idx]
        zmax_arr, e4_arr, labels = self.data[sub_idx]
        zmax_x = zmax_arr[i]
        e4_x   = e4_arr[i]
        y      = int(labels[i])
        return torch.FloatTensor(zmax_x), torch.FloatTensor(e4_x), torch.tensor(y, dtype=torch.long)


print("\nBuilding datasets...")
print("Train:")
train_ds = WearableDataset(TRAIN_SUBS, STUDENT_DATA_PATH)
print("Val:")
val_ds   = WearableDataset(VAL_SUBS, STUDENT_DATA_PATH)
print("Test:")
test_ds  = WearableDataset(TEST_SUBS, STUDENT_DATA_PATH)

for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    if len(ds) == 0:
        raise RuntimeError(f"{name}_ds has 0 samples! Check STUDENT_DATA_PATH.")
print("Datasets ready.")


# ============================================================
# ENCODERS (same lightweight design as the KD student)
# ============================================================
class ZmaxEEGEncoder(nn.Module):
    """Compact multi-scale CNN + BiGRU for raw Zmax EEG (2ch, 1920 samples @ 64Hz)."""
    def __init__(self, in_ch=2, d_model=D_MODEL, dropout=DROPOUT):
        super().__init__()
        mid = d_model // 2

        def branch(kernel):
            return nn.Sequential(
                nn.Conv1d(in_ch, mid, kernel_size=kernel, stride=4, padding=kernel // 2),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2), nn.Dropout(dropout)
            )
        self.small, self.large = branch(15), branch(60)
        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, 1920)
            L_s, L_l = self.small(dummy).shape[2], self.large(dummy).shape[2]
        target_L = min(L_s, L_l)
        self.pool_s = nn.AdaptiveAvgPool1d(target_L)
        self.pool_l = nn.AdaptiveAvgPool1d(target_L)
        self.proj = nn.Sequential(nn.Conv1d(2 * mid, d_model, kernel_size=1),
                                   nn.BatchNorm1d(d_model), nn.GELU())
        self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        fs, fl = self.pool_s(self.small(x)), self.pool_l(self.large(x))
        feat = self.proj(torch.cat([fs, fl], dim=1)).permute(0, 2, 1)
        gru_out, _ = self.gru(feat)
        return self.norm(feat.mean(dim=1) + gru_out.mean(dim=1))


class E4Encoder(nn.Module):
    """1D-CNN + BiGRU for E4 (3ch: BVP, HR, TEMP, 1920 samples @ 64Hz)."""
    def __init__(self, in_ch=3, d_model=D_MODEL, dropout=DROPOUT):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, d_model // 2, kernel_size=15, stride=4, padding=7),
            nn.BatchNorm1d(d_model // 2), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Conv1d(d_model // 2, d_model, kernel_size=8, padding=4),
            nn.BatchNorm1d(d_model), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Dropout(dropout),
        )
        self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        feat = self.conv(x).permute(0, 2, 1)
        gru_out, _ = self.gru(feat)
        return self.norm(feat.mean(dim=1) + gru_out.mean(dim=1))


# ============================================================
# S1 MODEL: simple concat fusion (NO gate, NO artifact info)
# ============================================================
class WearableBaselineModel(nn.Module):
    def __init__(self, d_model=D_MODEL, n_classes=5, dropout=DROPOUT):
        super().__init__()
        self.zmax_enc = ZmaxEEGEncoder(in_ch=2, d_model=d_model, dropout=dropout)
        self.e4_enc   = E4Encoder(in_ch=3, d_model=d_model, dropout=dropout)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model * 2), nn.Linear(d_model * 2, 64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, n_classes)
        )

    def forward(self, zmax_x, e4_x):
        z = self.zmax_enc(zmax_x)
        e = self.e4_enc(e4_x)
        fused = torch.cat([z, e], dim=1)   # simple concatenation, no gating
        logits = self.classifier(fused)
        return logits


# ============================================================
# CLASS WEIGHTS -- from TRAIN only
# ============================================================
cw_np = train_ds.label_counts.sum() / (5 * train_ds.label_counts)
cw = torch.FloatTensor(cw_np).to(device)
print(f"\nClass weights (from TRAIN set only):")
for name, w in zip(LABEL_NAMES, cw_np):
    print(f"  {name}: {w:.3f}")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_epoch_fn(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0
    preds, labs = [], []
    for zmax_x, e4_x, y in loader:
        zmax_x, e4_x, y = zmax_x.to(device), e4_x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(zmax_x, e4_x)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(y.cpu().numpy())
    n = len(loader)
    acc = accuracy_score(labs, preds)
    f1 = f1_score(labs, preds, average='macro', zero_division=0)
    return total_loss / n, acc, f1


@torch.no_grad()
def evaluate_fn(model, loader):
    model.eval()
    preds, labs = [], []
    for zmax_x, e4_x, y in loader:
        logits = model(zmax_x.to(device), e4_x.to(device))
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(y.numpy())
    preds = np.array(preds); labs = np.array(labs)
    acc = accuracy_score(labs, preds)
    f1 = f1_score(labs, preds, average='macro', zero_division=0)
    kappa = cohen_kappa_score(labs, preds)
    per_cls = f1_score(labs, preds, average=None, zero_division=0, labels=[0, 1, 2, 3, 4])
    return acc, f1, kappa, per_cls


# ============================================================
# CSV SETUP
# ============================================================
epoch_csv_path = os.path.join(EVAL_PATH, "epoch_log.csv")
epoch_fields = [
    "seed", "epoch", "train_loss", "train_acc", "train_f1_macro",
    "val_acc", "val_f1_macro", "val_kappa",
    "val_f1_Wake", "val_f1_N1", "val_f1_N2", "val_f1_N3", "val_f1_REM",
    "lr", "is_best",
]
with open(epoch_csv_path, 'w', newline='') as f:
    csv.DictWriter(f, epoch_fields).writeheader()

csv_summary_path = os.path.join(EVAL_PATH, "summary.csv")
summary_fields = [
    "seed", "best_epoch", "best_val_f1",
    "test_acc", "test_f1_macro", "test_kappa",
    "test_f1_Wake", "test_f1_N1", "test_f1_N2", "test_f1_N3", "test_f1_REM",
]
with open(csv_summary_path, 'w', newline='') as f:
    csv.DictWriter(f, summary_fields).writeheader()


# ============================================================
# 5-SEED TRAINING LOOP
# ============================================================
all_results = []

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}\nSEED {seed}  ({seed_idx+1}/{len(SEEDS)})\n{'='*60}")
    set_seed(seed)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
        generator=torch.Generator().manual_seed(seed)
    )
    val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = WearableBaselineModel().to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters : {n_params:,}")

    criterion = nn.CrossEntropyLoss(weight=cw)

    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=3e-4,
                             betas=(0.9, 0.98), eps=1e-9)
    total_steps = N_EPOCHS * len(train_loader)
    warmup_steps = int(0.05 * total_steps)

    def warmup_cosine(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        t = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1 + math.cos(math.pi * t))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_cosine)

    best_val_f1 = 0.0
    best_epoch  = -1
    best_path = os.path.join(EVAL_PATH, f"best_seed{seed}.pt")

    for epoch in range(1, N_EPOCHS + 1):
        tr_loss, tr_acc, tr_f1 = train_epoch_fn(
            model, train_loader, optimizer, scheduler, criterion
        )
        # === VAL -- used for checkpoint selection ===
        vl_acc, vl_f1, vl_kap, vl_per = evaluate_fn(model, val_loader)

        is_best = 0
        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            best_epoch  = epoch
            torch.save(model.state_dict(), best_path)
            is_best = 1

        lr = optimizer.param_groups[0]['lr']
        tag = " <- BEST" if is_best else ""
        print(f"  Ep[{epoch:02d}/{N_EPOCHS}] Loss:{tr_loss:.3f} "
              f"TrAcc:{tr_acc:.3f} TrF1:{tr_f1:.3f} | "
              f"ValAcc:{vl_acc:.3f} ValF1:{vl_f1:.3f} Valk:{vl_kap:.3f} "
              f"LR:{lr:.2e}{tag}")

        with open(epoch_csv_path, 'a', newline='') as f:
            csv.DictWriter(f, epoch_fields).writerow({
                "seed": seed, "epoch": epoch,
                "train_loss": round(tr_loss, 6),
                "train_acc": round(tr_acc, 6),
                "train_f1_macro": round(tr_f1, 6),
                "val_acc": round(vl_acc, 6),
                "val_f1_macro": round(vl_f1, 6),
                "val_kappa": round(vl_kap, 6),
                "val_f1_Wake": round(vl_per[0], 6),
                "val_f1_N1":   round(vl_per[1], 6),
                "val_f1_N2":   round(vl_per[2], 6),
                "val_f1_N3":   round(vl_per[3], 6),
                "val_f1_REM":  round(vl_per[4], 6),
                "lr": lr,
                "is_best": is_best,
            })

    # === FINAL, ONE-TIME TEST EVALUATION ===
    model.load_state_dict(torch.load(best_path, map_location=device))
    fin_acc, fin_f1, fin_kap, fin_per = evaluate_fn(model, test_loader)

    print(f"\n  Seed {seed}: best val F1={best_val_f1:.4f} at epoch {best_epoch}")
    print(f"  Seed {seed} FINAL TEST (evaluated once): "
          f"Acc={fin_acc*100:.2f}% F1={fin_f1:.4f} k={fin_kap:.4f}")
    for i, name in enumerate(LABEL_NAMES):
        print(f"    {name:6s}: {fin_per[i]:.4f}")

    all_results.append({
        'seed': seed, 'acc': fin_acc, 'f1': fin_f1, 'kappa': fin_kap,
        'per_cls': fin_per, 'best_epoch': best_epoch, 'best_val_f1': best_val_f1
    })

    with open(csv_summary_path, 'a', newline='') as f:
        csv.DictWriter(f, summary_fields).writerow({
            "seed": seed, "best_epoch": best_epoch, "best_val_f1": round(best_val_f1, 4),
            "test_acc": round(fin_acc, 4), "test_f1_macro": round(fin_f1, 4),
            "test_kappa": round(fin_kap, 4),
            "test_f1_Wake": round(fin_per[0], 4), "test_f1_N1": round(fin_per[1], 4),
            "test_f1_N2": round(fin_per[2], 4), "test_f1_N3": round(fin_per[3], 4),
            "test_f1_REM": round(fin_per[4], 4),
        })

# ============================================================
# FINAL REPORT
# ============================================================
accs   = np.array([r['acc'] for r in all_results]) * 100
f1s    = np.array([r['f1'] for r in all_results])
kappas = np.array([r['kappa'] for r in all_results])

print(f"\n{'='*60}\nS1: WEARABLE-ONLY BASELINE (VAL-SELECTED, HONEST TEST) -- 5-SEED REPORT\n{'='*60}")
print(f"Accuracy : {accs.mean():.2f} +- {accs.std():.2f}%")
print(f"Macro F1 : {f1s.mean():.4f} +- {f1s.std():.4f}")
print(f"Kappa    : {kappas.mean():.4f} +- {kappas.std():.4f}")

print("\nThis is your S1 reference point. Next step (S2) adds")
print("knowledge distillation from the frozen PSG teacher and we")
print("compare against this baseline to measure the benefit of KD alone.")
print("NOTE: this is a re-run on the NEW val-fixed split, so the numbers")
print("will differ from your original S1 (57.09%/0.5112) -- re-run S2-S5")
print("on this same split before comparing across scripts.")

print(f"\nEpoch log : {epoch_csv_path}")
print(f"Summary   : {csv_summary_path}")
print("Done!")

Device : cuda
S1: Wearable-only baseline (Zmax EEG + E4, plain CE, no KD, no gate)
Output : D:\22\AA\AA journal\evaluation\teacher-student\s1_wearable_baseline_valfixed
Split loaded -> Train:65  Val:11  Test:20

Building datasets...
Train:
  Subjects: 61   Samples: 57,021
Val:
  Subjects: 10   Samples: 9,745
Test:
  Subjects: 19   Samples: 18,899
Datasets ready.

Class weights (from TRAIN set only):
  Wake: 2.094
  N1: 3.192
  N2: 0.436
  N3: 1.000
  REM: 1.092

SEED 42  (1/5)
  Parameters : 153,989
  Ep[01/30] Loss:1.343 TrAcc:0.443 TrF1:0.396 | ValAcc:0.475 ValF1:0.449 Valk:0.341 LR:3.33e-04 <- BEST
  Ep[02/30] Loss:1.161 TrAcc:0.533 TrF1:0.486 | ValAcc:0.500 ValF1:0.454 Valk:0.353 LR:5.00e-04 <- BEST
  Ep[03/30] Loss:1.095 TrAcc:0.568 TrF1:0.520 | ValAcc:0.511 ValF1:0.484 Valk:0.386 LR:4.97e-04 <- BEST
  Ep[04/30] Loss:1.046 TrAcc:0.587 TrF1:0.539 | ValAcc:0.558 ValF1:0.519 Valk:0.435 LR:4.91e-04 <- BEST
  Ep[05/30] Loss:1.008 TrAcc:0.603 TrF1:0.558 | ValAcc:0.567 ValF1:0.534 Valk:0